In [1]:
%pip install pygame

   ---------------------------------------- 0.0/10.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.6 MB ? eta -:--:--
    --------------------------------------- 0.3/10.6 MB ? eta -:--:--
   - -------------------------------------- 0.5/10.6 MB 1.4 MB/s eta 0:00:08
   -- ------------------------------------- 0.8/10.6 MB 1.4 MB/s eta 0:00:08
   --- ------------------------------------ 1.0/10.6 MB 1.4 MB/s eta 0:00:08
   ---- ----------------------------------- 1.3/10.6 MB 1.4 MB/s eta 0:00:07
   ------ --------------------------------- 1.8/10.6 MB 1.4 MB/s eta 0:00:07
   --------- ------------------------------ 2.6/10.6 MB 1.8 MB/s eta 0:00:05
   ----------- ---------------------------- 3.1/10.6 MB 1.9 MB/s eta 0:00:04
   ------------- -------------------------- 3.7/10.6 MB 1.9 MB/s eta 0:00:04
   --------------- ------------------------ 4.2/10.6 MB 2.1 MB/s eta 0:00:04
   --------------------- ------------------ 5.8/10.6 MB 2.6 MB/s eta 0:00:02
   -----------------


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import pygame
import random
import numpy as np
from collections import deque
import torch
import torch.nn as nn
import torch.optim as optim

# -----------------------------
# CONFIG
# -----------------------------
WIDTH, HEIGHT = 400, 600
PIPE_WIDTH, PIPE_GAP = 120, 160
GRAVITY = 0.5
JUMP_VELOCITY = -8
FPS = 60
GAMMA = 0.95
LR = 0.001
MEM_SIZE = 100_000
BATCH_SIZE = 2000
MAX_EPISODES = 300
DECISION_INTERVAL = 5
EXPLORE_EPISODES = 100  # 👈 Explore for 100 episodes, then exploit

# -----------------------------
# ENVIRONMENT
# -----------------------------
class FlappyBirdEnv:
    def __init__(self):
        pygame.init()
        self.display = pygame.display.set_mode((WIDTH, HEIGHT))
        pygame.display.set_caption("Flappy Bird RL - Explore/Exploit Mode")
        self.clock = pygame.time.Clock()
        self.reset()

    def reset(self):
        self.bird_y = HEIGHT // 2
        self.bird_v = 0
        self.pipe_x = WIDTH
        self.pipe_gap_y = random.randint(180, HEIGHT - 180)
        self.score = 0
        self.frame = 0
        self.done = False
        return self.get_state()

    def step(self, action):
        reward = 0.1  # survival reward
        self.frame += 1

        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                pygame.quit(); quit()

        if action == 1:
            self.bird_v = JUMP_VELOCITY

        self.bird_v += GRAVITY
        self.bird_y += self.bird_v
        self.pipe_x -= 4

        if self.pipe_x + PIPE_WIDTH < 0:
            self.pipe_x = WIDTH
            self.pipe_gap_y = random.randint(150, HEIGHT - 150)
            self.score += 1
            reward += 3.0

        gap_top = self.pipe_gap_y - PIPE_GAP // 2
        gap_bottom = self.pipe_gap_y + PIPE_GAP // 2
        center = self.pipe_gap_y
        dist = abs(self.bird_y - center)

        if self.pipe_x < 60 < self.pipe_x + PIPE_WIDTH:
            if gap_top < self.bird_y < gap_bottom:
                reward += 2.0
            else:
                reward -= 3.0
        else:
            reward += max(0, 1 - dist / 300) * 1.0

        if self.bird_y < 30 or self.bird_y > HEIGHT - 30:
            reward -= 2.5

        if (self.bird_y <= 0 or self.bird_y >= HEIGHT or
            (self.pipe_x < 60 < self.pipe_x + PIPE_WIDTH and
             (self.bird_y < gap_top or self.bird_y > gap_bottom))):
            reward = -10
            self.done = True

        self.render()
        self.clock.tick(FPS)
        return self.get_state(), reward, self.done, self.score

    def get_state(self):
        return np.array([
            self.bird_y / HEIGHT,
            self.bird_v / 10,
            self.pipe_x / WIDTH,
            self.pipe_gap_y / HEIGHT
        ], dtype=np.float32)

    def render(self):
        self.display.fill((135, 206, 235))
        pygame.draw.rect(self.display, (0, 200, 0),
                         (self.pipe_x, 0, PIPE_WIDTH, self.pipe_gap_y - PIPE_GAP // 2))
        pygame.draw.rect(self.display, (0, 200, 0),
                         (self.pipe_x, self.pipe_gap_y + PIPE_GAP // 2, PIPE_WIDTH, HEIGHT))
        pygame.draw.rect(self.display, (200, 255, 200),
                         (self.pipe_x, self.pipe_gap_y - PIPE_GAP // 2, PIPE_WIDTH, PIPE_GAP), 2)
        pygame.draw.circle(self.display, (255, 255, 0), (60, int(self.bird_y)), 10)
        pygame.display.flip()

# -----------------------------
# DQN
# -----------------------------
class DQN(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 128)
        self.fc2 = nn.Linear(128, 128)
        self.fc3 = nn.Linear(128, output_dim)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)

class Agent:
    def __init__(self):
        self.model = DQN(4, 2)
        self.target_model = DQN(4, 2)
        self.target_model.load_state_dict(self.model.state_dict())
        self.optimizer = optim.Adam(self.model.parameters(), lr=LR)
        self.criterion = nn.MSELoss()
        self.memory = deque(maxlen=MEM_SIZE)
        self.epsilon = 1.0
        self.steps = 0

    def act(self, state):
        if random.random() < self.epsilon:
            return random.randint(0, 1)
        state_t = torch.tensor(state, dtype=torch.float32).unsqueeze(0)
        with torch.no_grad():
            q_values = self.model(state_t)
        return torch.argmax(q_values).item()

    def remember(self, s, a, r, s2, done):
        self.memory.append((s, a, r, s2, done))

    def train(self):
        if len(self.memory) < BATCH_SIZE:
            return
        batch = random.sample(self.memory, BATCH_SIZE)
        states, actions, rewards, next_states, dones = zip(*batch)
        states = torch.tensor(states, dtype=torch.float32)
        actions = torch.tensor(actions, dtype=torch.long)
        rewards = torch.tensor(rewards, dtype=torch.float32)
        next_states = torch.tensor(next_states, dtype=torch.float32)
        dones = torch.tensor(dones, dtype=torch.float32)

        q_pred = self.model(states).gather(1, actions.unsqueeze(1)).squeeze(1)
        with torch.no_grad():
            q_next = self.target_model(next_states).max(1)[0]
            q_target = rewards + GAMMA * q_next * (1 - dones)

        loss = self.criterion(q_pred, q_target)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        tau = 0.01
        for target_param, param in zip(self.target_model.parameters(), self.model.parameters()):
            target_param.data.copy_(tau * param.data + (1.0 - tau) * target_param.data)

# -----------------------------
# TRAIN LOOP
# -----------------------------
def train():
    env = FlappyBirdEnv()
    agent = Agent()
    best_score = 0

    for episode in range(1, MAX_EPISODES + 1):
        state = env.reset()
        total_reward = 0
        done = False
        frame_count = 0
        action = 0

        # --- Exploration vs Exploitation Logic ---
        if episode <= EXPLORE_EPISODES:
            agent.epsilon = 0.5  # explore more
        else:
            agent.epsilon = 0.05  # exploit learned policy

        while not done:
            frame_count += 1
            if frame_count % DECISION_INTERVAL == 0:
                action = agent.act(state)

            next_state, reward, done, score = env.step(action)
            agent.remember(state, action, reward, next_state, done)
            agent.train()
            state = next_state
            total_reward += reward

        best_score = max(best_score, score)
        print(f"Ep {episode:03d} | Mode: {'Explore' if episode <= EXPLORE_EPISODES else 'Exploit'} "
              f"| Score: {score} | Best: {best_score} | Reward: {total_reward:.2f}")

if __name__ == "__main__":
    train()


pygame 2.6.1 (SDL 2.28.4, Python 3.13.5)
Hello from the pygame community. https://www.pygame.org/contribute.html
Ep 001 | Mode: Explore | Score: 0 | Best: 0 | Reward: 23.79
Ep 002 | Mode: Explore | Score: 0 | Best: 0 | Reward: -1.23
Ep 003 | Mode: Explore | Score: 0 | Best: 0 | Reward: 18.29
Ep 004 | Mode: Explore | Score: 0 | Best: 0 | Reward: 1.26
Ep 005 | Mode: Explore | Score: 0 | Best: 0 | Reward: -0.91
Ep 006 | Mode: Explore | Score: 0 | Best: 0 | Reward: 15.42
Ep 007 | Mode: Explore | Score: 0 | Best: 0 | Reward: 14.83
Ep 008 | Mode: Explore | Score: 0 | Best: 0 | Reward: 16.48
Ep 009 | Mode: Explore | Score: 0 | Best: 0 | Reward: -0.17
Ep 010 | Mode: Explore | Score: 0 | Best: 0 | Reward: 5.23
Ep 011 | Mode: Explore | Score: 0 | Best: 0 | Reward: 12.24
Ep 012 | Mode: Explore | Score: 0 | Best: 0 | Reward: 4.40
Ep 013 | Mode: Explore | Score: 0 | Best: 0 | Reward: 1.43
Ep 014 | Mode: Explore | Score: 0 | Best: 0 | Reward: 2.17
Ep 015 | Mode: Explore | Score: 0 | Best: 0 | Reward

C:\Users\harsh\AppData\Local\Temp\ipykernel_21220\3719088019.py:154: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:256.)
  states = torch.tensor(states, dtype=torch.float32)


Ep 039 | Mode: Explore | Score: 0 | Best: 0 | Reward: -1.28
Ep 040 | Mode: Explore | Score: 0 | Best: 0 | Reward: 30.32
Ep 041 | Mode: Explore | Score: 0 | Best: 0 | Reward: 27.39
Ep 042 | Mode: Explore | Score: 0 | Best: 0 | Reward: 9.79
Ep 043 | Mode: Explore | Score: 0 | Best: 0 | Reward: 4.42
Ep 044 | Mode: Explore | Score: 0 | Best: 0 | Reward: 14.92
Ep 045 | Mode: Explore | Score: 0 | Best: 0 | Reward: 45.17
Ep 046 | Mode: Explore | Score: 0 | Best: 0 | Reward: 48.86
Ep 047 | Mode: Explore | Score: 0 | Best: 0 | Reward: -14.20
Ep 048 | Mode: Explore | Score: 0 | Best: 0 | Reward: 53.02
Ep 049 | Mode: Explore | Score: 0 | Best: 0 | Reward: 10.14
Ep 050 | Mode: Explore | Score: 0 | Best: 0 | Reward: 35.71
Ep 051 | Mode: Explore | Score: 0 | Best: 0 | Reward: 69.24
Ep 052 | Mode: Explore | Score: 0 | Best: 0 | Reward: 20.50
Ep 053 | Mode: Explore | Score: 0 | Best: 0 | Reward: 32.64
Ep 054 | Mode: Explore | Score: 0 | Best: 0 | Reward: 52.43
Ep 055 | Mode: Explore | Score: 0 | Best:

KeyboardInterrupt: 

: 

In [ ]:
import pygame
import random
import numpy as np
from collections import deque
import torch
import torch.nn as nn
import torch.optim as optim

# -----------------------------
# CONFIG
# -----------------------------
WIDTH, HEIGHT = 400, 600
PIPE_WIDTH, PIPE_GAP = 120, 160
GRAVITY = 0.5
JUMP_VELOCITY = -8
FPS = 60
GAMMA = 0.95
LR = 0.001
MEM_SIZE = 100_000
BATCH_SIZE = 2000
MAX_EPISODES = 300
DECISION_INTERVAL = 5
EXPLORE_EPISODES = 100

# -----------------------------
# ENVIRONMENT
# -----------------------------
class FlappyBirdEnv:
    def __init__(self):
        pygame.init()
        self.display = pygame.display.set_mode((WIDTH, HEIGHT))
        pygame.display.set_caption("Flappy Bird RL - With Score")
        self.clock = pygame.time.Clock()
        self.font = pygame.font.SysFont("Arial", 32, bold=True)
        self.reset()

    def reset(self):
        self.bird_y = HEIGHT // 2
        self.bird_v = 0
        self.pipe_x = WIDTH
        self.pipe_gap_y = random.randint(180, HEIGHT - 180)
        self.score = 0
        self.frame = 0
        self.done = False
        return self.get_state()

    def step(self, action):
        reward = 0.1
        self.frame += 1

        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                pygame.quit(); quit()

        if action == 1:
            self.bird_v = JUMP_VELOCITY

        self.bird_v += GRAVITY
        self.bird_y += self.bird_v
        self.pipe_x -= 4

        if self.pipe_x + PIPE_WIDTH < 0:
            self.pipe_x = WIDTH
            self.pipe_gap_y = random.randint(150, HEIGHT - 150)
            self.score += 1
            reward += 3.0

        gap_top = self.pipe_gap_y - PIPE_GAP // 2
        gap_bottom = self.pipe_gap_y + PIPE_GAP // 2
        center = self.pipe_gap_y
        dist = abs(self.bird_y - center)

        # Reward zones (your P/R layout logic)
        if self.pipe_x < 60 < self.pipe_x + PIPE_WIDTH:
            if gap_top < self.bird_y < gap_bottom:
                reward += 2.0
            else:
                reward -= 3.0
        else:
            reward += max(0, 1 - dist / 300) * 1.0

        if self.bird_y < 30 or self.bird_y > HEIGHT - 30:
            reward -= 2.5

        if (self.bird_y <= 0 or self.bird_y >= HEIGHT or
            (self.pipe_x < 60 < self.pipe_x + PIPE_WIDTH and
             (self.bird_y < gap_top or self.bird_y > gap_bottom))):
            reward = -10
            self.done = True

        self.render()
        self.clock.tick(FPS)
        return self.get_state(), reward, self.done, self.score

    def get_state(self):
        return np.array([
            self.bird_y / HEIGHT,
            self.bird_v / 10,
            self.pipe_x / WIDTH,
            self.pipe_gap_y / HEIGHT
        ], dtype=np.float32)

    def render(self):
        self.display.fill((135, 206, 235))
        # Pipes
        pygame.draw.rect(self.display, (0, 200, 0),
                         (self.pipe_x, 0, PIPE_WIDTH, self.pipe_gap_y - PIPE_GAP // 2))
        pygame.draw.rect(self.display, (0, 200, 0),
                         (self.pipe_x, self.pipe_gap_y + PIPE_GAP // 2, PIPE_WIDTH, HEIGHT))
        # Reward zone outline
        pygame.draw.rect(self.display, (200, 255, 200),
                         (self.pipe_x, self.pipe_gap_y - PIPE_GAP // 2, PIPE_WIDTH, PIPE_GAP), 2)
        # Bird
        pygame.draw.circle(self.display, (255, 255, 0), (60, int(self.bird_y)), 10)

        # ✅ SCORE DISPLAY
        score_text = self.font.render(f"Score: {self.score}", True, (0, 0, 0))
        self.display.blit(score_text, (10, 10))

        pygame.display.flip()

# -----------------------------
# DQN AGENT
# -----------------------------
class DQN(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 128)
        self.fc2 = nn.Linear(128, 128)
        self.fc3 = nn.Linear(128, output_dim)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)

class Agent:
    def __init__(self):
        self.model = DQN(4, 2)
        self.target_model = DQN(4, 2)
        self.target_model.load_state_dict(self.model.state_dict())
        self.optimizer = optim.Adam(self.model.parameters(), lr=LR)
        self.criterion = nn.MSELoss()
        self.memory = deque(maxlen=MEM_SIZE)
        self.epsilon = 1.0

    def act(self, state):
        if random.random() < self.epsilon:
            return random.randint(0, 1)
        state_t = torch.tensor(state, dtype=torch.float32).unsqueeze(0)
        with torch.no_grad():
            q_values = self.model(state_t)
        return torch.argmax(q_values).item()

    def remember(self, s, a, r, s2, done):
        self.memory.append((s, a, r, s2, done))

    def train(self):
        if len(self.memory) < BATCH_SIZE:
            return
        batch = random.sample(self.memory, BATCH_SIZE)
        states, actions, rewards, next_states, dones = zip(*batch)
        states = torch.tensor(states, dtype=torch.float32)
        actions = torch.tensor(actions, dtype=torch.long)
        rewards = torch.tensor(rewards, dtype=torch.float32)
        next_states = torch.tensor(next_states, dtype=torch.float32)
        dones = torch.tensor(dones, dtype=torch.float32)

        q_pred = self.model(states).gather(1, actions.unsqueeze(1)).squeeze(1)
        with torch.no_grad():
            q_next = self.target_model(next_states).max(1)[0]
            q_target = rewards + GAMMA * q_next * (1 - dones)

        loss = self.criterion(q_pred, q_target)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        tau = 0.01
        for target_param, param in zip(self.target_model.parameters(), self.model.parameters()):
            target_param.data.copy_(tau * param.data + (1.0 - tau) * target_param.data)

# -----------------------------
# TRAIN LOOP
# -----------------------------
def train():
    env = FlappyBirdEnv()
    agent = Agent()
    best_score = 0

    for episode in range(1, MAX_EPISODES + 1):
        state = env.reset()
        total_reward = 0
        done = False
        frame_count = 0
        action = 0

        # Explore → Exploit
        if episode <= EXPLORE_EPISODES:
            agent.epsilon = 0.5
        else:
            agent.epsilon = 0.05

        while not done:
            frame_count += 1
            if frame_count % DECISION_INTERVAL == 0:
                action = agent.act(state)

            next_state, reward, done, score = env.step(action)
            agent.remember(state, action, reward, next_state, done)
            agent.train()
            state = next_state
            total_reward += reward

        best_score = max(best_score, score)
        print(f"Ep {episode:03d} | Mode: {'Explore' if episode <= EXPLORE_EPISODES else 'Exploit'} "
              f"| Score: {score} | Best: {best_score} | Reward: {total_reward:.2f}")

if __name__ == "__main__":
    train()
s

pygame 2.6.1 (SDL 2.28.4, Python 3.13.5)
Hello from the pygame community. https://www.pygame.org/contribute.html
Ep 001 | Mode: Explore | Score: 0 | Best: 0 | Reward: -3.80
Ep 002 | Mode: Explore | Score: 0 | Best: 0 | Reward: 10.69
Ep 003 | Mode: Explore | Score: 0 | Best: 0 | Reward: 2.57
Ep 004 | Mode: Explore | Score: 0 | Best: 0 | Reward: 17.36
Ep 005 | Mode: Explore | Score: 0 | Best: 0 | Reward: 18.11
Ep 006 | Mode: Explore | Score: 0 | Best: 0 | Reward: 13.31
Ep 007 | Mode: Explore | Score: 0 | Best: 0 | Reward: 17.05
Ep 008 | Mode: Explore | Score: 0 | Best: 0 | Reward: 29.55
Ep 009 | Mode: Explore | Score: 0 | Best: 0 | Reward: -2.47
Ep 010 | Mode: Explore | Score: 0 | Best: 0 | Reward: 23.09
Ep 011 | Mode: Explore | Score: 0 | Best: 0 | Reward: 15.69
Ep 012 | Mode: Explore | Score: 0 | Best: 0 | Reward: 4.50
Ep 013 | Mode: Explore | Score: 0 | Best: 0 | Reward: 23.46
Ep 014 | Mode: Explore | Score: 0 | Best: 0 | Reward: 12.58
Ep 015 | Mode: Explore | Score: 0 | Best: 0 | Rew

C:\Users\harsh\AppData\Local\Temp\ipykernel_18700\498625498.py:163: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:256.)
  states = torch.tensor(states, dtype=torch.float32)


Ep 041 | Mode: Explore | Score: 0 | Best: 0 | Reward: -20.68
Ep 042 | Mode: Explore | Score: 0 | Best: 0 | Reward: 6.77
Ep 043 | Mode: Explore | Score: 0 | Best: 0 | Reward: 45.20
Ep 044 | Mode: Explore | Score: 0 | Best: 0 | Reward: 29.04
Ep 045 | Mode: Explore | Score: 0 | Best: 0 | Reward: 20.46
Ep 046 | Mode: Explore | Score: 0 | Best: 0 | Reward: 28.31
Ep 047 | Mode: Explore | Score: 0 | Best: 0 | Reward: -7.78
Ep 048 | Mode: Explore | Score: 0 | Best: 0 | Reward: 12.23
Ep 049 | Mode: Explore | Score: 0 | Best: 0 | Reward: -3.72
Ep 050 | Mode: Explore | Score: 0 | Best: 0 | Reward: 42.76
Ep 051 | Mode: Explore | Score: 0 | Best: 0 | Reward: 14.22
Ep 052 | Mode: Explore | Score: 0 | Best: 0 | Reward: 20.69
Ep 053 | Mode: Explore | Score: 0 | Best: 0 | Reward: 29.37
Ep 054 | Mode: Explore | Score: 0 | Best: 0 | Reward: 50.83
Ep 055 | Mode: Explore | Score: 0 | Best: 0 | Reward: 24.48
Ep 056 | Mode: Explore | Score: 0 | Best: 0 | Reward: 76.48
Ep 057 | Mode: Explore | Score: 0 | Best

error: display Surface quit

: 